# 🧠 SpikeLogBERT — HDFS Full Dataset Training

Training pipeline on the full HDFS dataset (~11M logs, ~29 templates).

> ⚠️ Requires Colab Pro or A100 GPU for reasonable training time.

---

## 0. GPU Check

In [ ]:
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("⚠️ No GPU! Go to Runtime → Change runtime type → T4/A100")

## 1. Setup

In [ ]:
!git clone https://github.com/thuanbui1309/spikeLogBert.git
%cd spikeLogBert
!pip install spikingjelly scikit-learn pandas tqdm pyyaml -q

## 2. Download HDFS Full Dataset

The full HDFS dataset (~1.5GB) is available from Zenodo via LogHub-2.0.
If you have it on Google Drive, mount and copy instead.

In [ ]:
# Option A: Download HDFS 2k for quick test (uncomment to use)
# !python data/download.py --dataset HDFS --output data/raw

# Option B: Use full HDFS dataset from Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# !cp /content/drive/MyDrive/datasets/HDFS.log_structured.csv data/raw/

# Option C: Download full HDFS from Zenodo (large, ~1.5GB)
# !pip install zenodo_get -q
# !zenodo_get 8196385 -o data/raw/

# For now, use the 2k version
!python data/download.py --dataset HDFS --output data/raw

In [ ]:
import os

# Detect which file we have
full_csv = 'data/raw/HDFS.log_structured.csv'
small_csv = 'data/raw/HDFS_2k.log_structured.csv'

if os.path.exists(full_csv):
    INPUT_CSV = full_csv
    print(f"Using FULL dataset: {INPUT_CSV}")
else:
    INPUT_CSV = small_csv
    print(f"Using 2k dataset: {INPUT_CSV}")

# Optional: subsample for faster iteration
# MAX_SAMPLES = 100000  # 100k for testing, None for full
MAX_SAMPLES = None

In [ ]:
# Preprocess
cmd = f"python data/dataset.py --input {INPUT_CSV} --output data/hdfs"
if MAX_SAMPLES:
    cmd += f" --max_samples {MAX_SAMPLES}"
!{cmd}

In [ ]:
# Count labels
with open('data/hdfs/label_mapping.txt') as f:
    lines = f.readlines()
NUM_LABELS = len(lines)
print(f"Templates: {NUM_LABELS}")
for line in lines:
    print(f"  {line.strip()}")

!echo "---"
!wc -l data/hdfs/train.txt data/hdfs/val.txt data/hdfs/test.txt

## 3. Stage 1: Fine-tune BERT Teacher

In [ ]:
!python train_teacher.py \
    --dataset_dir data/hdfs \
    --label_num {NUM_LABELS} \
    --epochs 4 \
    --batch_size 32 \
    --lr 5e-5

## 4. Stage 2: Knowledge Distillation → SpikeLogBERT

In [ ]:
import os
TEACHER_PATH = f"saved_models/teacher/best_teacher_{NUM_LABELS}classes"
assert os.path.exists(TEACHER_PATH), f"Teacher not found: {TEACHER_PATH}"
print(f"Teacher: {TEACHER_PATH}")

In [ ]:
!python distill.py \
    --dataset_dir data/hdfs \
    --teacher_model_path {TEACHER_PATH} \
    --label_num {NUM_LABELS} \
    --depths 6 \
    --dim 768 \
    --num_step 4 \
    --epochs 30 \
    --batch_size 8 \
    --lr 5e-4

## 5. Stage 3: Evaluation

In [ ]:
import glob
distilled_models = sorted(glob.glob("saved_models/distilled/spikelogbert_*.pth"))
if distilled_models:
    BEST_MODEL = distilled_models[-1]
    print(f"Best model: {BEST_MODEL}")
else:
    print("⚠️ No distilled model found.")

In [ ]:
!python evaluate.py \
    --model_path {BEST_MODEL} \
    --dataset_dir data/hdfs \
    --label_num {NUM_LABELS} \
    --num_step 4

In [ ]:
import json
with open('results/eval_test.json') as f:
    results = json.load(f)
print(f"Parsing Accuracy: {results['parsing_accuracy']:.4f}")
print(f"Macro F1: {results['macro_f1']:.4f}")
print(f"Weighted F1: {results['weighted_f1']:.4f}")

## 6. Save to Google Drive

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/spikeLogBert/
# !cp -r saved_models/ /content/drive/MyDrive/spikeLogBert/
# !cp -r results/ /content/drive/MyDrive/spikeLogBert/
# print("✅ Saved to Drive")